# Citi Bike Data Overview

This notebook summarizes the processed Jersey City Citi Bike demand files used by the rebalancing experiments. It is intended for exploratory analysis and report support, not for production preprocessing.


## Goals

- confirm that the monthly processed files load cleanly,
- inspect date coverage and hourly flow volume,
- identify the highest-activity stations,
- compare demand volume with the NOAA weather coverage window.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

processed_paths = sorted(Path('../data/processed').glob('jc_*.csv'))
weather_path = Path('../data/external/noaa_daily_usw00014734_20250101_20260228.csv')

monthly_frames = []
monthly_summary_rows = []
for path in processed_paths:
    frame = pd.read_csv(path)
    frame['hour'] = pd.to_datetime(frame['hour'])
    frame['source_file'] = path.name
    monthly_frames.append(frame)
    monthly_summary_rows.append(
        {
            'file': path.name,
            'flow_rows': len(frame),
            'episode_days': frame['hour'].dt.date.nunique(),
            'trip_count_sum': int(frame['trip_count'].sum()),
            'start_station_count': frame['start_station_id'].nunique(),
            'end_station_count': frame['end_station_id'].nunique(),
        }
    )

flows = pd.concat(monthly_frames, ignore_index=True)
monthly_summary = pd.DataFrame(monthly_summary_rows)
weather = pd.read_csv(weather_path)
weather['DATE'] = pd.to_datetime(weather['DATE'])

monthly_summary


In [ ]:
print('processed months:', len(processed_paths))
print('total flow rows:', len(flows))
print('total trip count:', int(flows['trip_count'].sum()))
print('unique hourly timestamps:', flows['hour'].nunique())
print('weather days:', weather['DATE'].dt.date.nunique())

monthly_summary[['file', 'flow_rows', 'episode_days', 'trip_count_sum']]


In [ ]:
station_activity = (
    flows.groupby('start_station_id')['trip_count'].sum()
    .add(flows.groupby('end_station_id')['trip_count'].sum(), fill_value=0)
    .sort_values(ascending=False)
    .rename('total_activity')
)

station_activity.head(10)


In [ ]:
monthly_trip_counts = (
    flows.assign(month=flows['hour'].dt.to_period('M').astype(str))
    .groupby('month', as_index=False)['trip_count']
    .sum()
)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(monthly_trip_counts['month'], monthly_trip_counts['trip_count'], color='#0b3954')
axes[0].set_title('Monthly trip count from processed flows')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Trips')

station_activity.head(10).sort_values().plot(kind='barh', ax=axes[1], color='#bf4e30')
axes[1].set_title('Top 10 stations by total activity')
axes[1].set_xlabel('Trips')

figure.tight_layout()
plt.show()


## Notes

- The processed monthly files are built for RL experiments, so a single monthly file can include boundary hours from the adjacent local day after timezone normalization.
- The primary report uses a five-station subset selected from the training window by total activity.
- If any exploratory transformation becomes part of the repeatable workflow, it should move into `scripts/` or `src/`.
